# Modelo Supervisado 2: XGBoost

Este modelo complementa el análisis abordando el objetivo de **clasificar el nivel de riesgo agrícola** (multiclase) de una siembra.
Se utiliza **XGBoost (Extreme Gradient Boosting)** por su robustez ante datos tabulares no lineales y su capacidad para manejar clases desbalanceadas mediante pesos (`sample_weight`).

## Fase 0: Configuración Inicial y Carga de Datos

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from src.data_loader import DataLoader
from src.xgboost_model import XGBoostDataPrep

# Configuraciones globales
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# Carga de datos
loader = DataLoader('../data/raw/siap_2010_2024.csv')
try:
    df_raw = loader.cargar()
except FileNotFoundError:
    # Por si se ejecuta donde el dataset procesado está directamente en la raíz
    loader = DataLoader('../siap_procesado.csv')
    df_raw = loader.cargar()

print("Dimensiones del dataset:", df_raw.shape)
df_raw.dropna(subset=['Sembrada', 'Volumenproduccion'], inplace=True)

## Fase 1: Ingeniería de Features y Preparación de Datos

Se aplicará el preprocesamiento definido en `src/xgboost_model.py`:
1. Codificación de variables (LabelEncoding).
2. Creación de variables derivadas (`log_sembrada`, `interaccion_mod_ciclo`).
3. **Split Temporal**: 
   - Entrenamiento: 2010-2020
   - Validación: 2021-2022
   - Test: 2023-2024
4. Cálculo de pesos para el desbalance de clases.

In [ ]:
prep = XGBoostDataPrep(df_raw)
df_processed = prep.preprocess()

(X_train, y_train), (X_val, y_val), (X_test, y_test) = prep.temporal_split()

# Calcular sample weights para entrenamiento debido al desbalance extremo
sample_weights = prep.get_sample_weights(y_train)